In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import BlipProcessor, BlipForConditionalGeneration
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge


MODEL_NAME = "Salesforce/blip-image-captioning-base"
CSV_PATH = "/msrvtt_2k_preprocessed.csv"
NPZ_DIR = "/final_frames_v2"
OUTPUT_REPORT = "baseline_evaluation_report.txt"

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

print(f"Loading ORIGINAL Pre-trained Model ({MODEL_NAME}) for Baseline Evaluation...")
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()


# Build Ground Truth Dictionary (gts)

df = pd.read_csv(CSV_PATH)
gts = {}
for _, row in df.iterrows():
    vid = row['video_id']
    cap = str(row['caption'])
    if vid not in gts:
        gts[vid] = []
    gts[vid].append(cap)

unique_videos = list(gts.keys())
print(f"Found {len(unique_videos)} unique videos to evaluate.")


# Generate Baseline Captions (res)

res = {}
print("Generating original AI captions for scoring...")
for vid in tqdm(unique_videos, desc="Evaluating Videos"):
    npz_path = os.path.join(NPZ_DIR, f"{vid}.npz")
    
    try:
        with np.load(npz_path) as data:
            video = data['video'] 
            
        middle_idx = video.shape[1] // 2
        frame = video[:, middle_idx, :, :] 
        
        frame = np.transpose(frame, (1, 2, 0))
        if frame.max() <= 1.0:
            frame = (frame * 255).astype(np.uint8)
        else:
            frame = frame.astype(np.uint8)
            
        inputs = processor(images=frame, return_tensors="pt").to(DEVICE)
        
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_length=30,
                num_beams=5,
                early_stopping=True,
                no_repeat_ngram_size=2
            )
            
        generated_caption = processor.decode(output_ids[0], skip_special_tokens=True)
        res[vid] = [generated_caption]
        
    except Exception as e:
        continue

# Calculate All Metrics

print("\nCalculating metrics...")

scorers = [
    (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
    (Rouge(), "ROUGE_L"),
    (Cider(), "CIDEr")
]

final_scores = {}

for scorer, method in scorers:
    try:
        score, scores = scorer.compute_score(gts, res)
        if type(method) == list:
            for sc, scs, m in zip(score, scores, method):
                final_scores[m] = sc * 100 
        else:
            final_scores[method] = score * 100
    except Exception as e:
        final_scores[method] = "Error"


# Save the Report

print("\n" + "="*40)
print("📊 BASELINE EVALUATION REPORT")
print("="*40)
for metric, val in final_scores.items():
    if isinstance(val, float):
        print(f"{metric}: {val:.2f}")
    else:
        print(f"{metric}: {val}")
print("="*40)

with open(OUTPUT_REPORT, "w") as f:
    f.write("BASELINE MODEL EVALUATION REPORT\n")
    f.write(f"Model: {MODEL_NAME}\n")
    f.write("="*40 + "\n")
    for metric, val in final_scores.items():
        if isinstance(val, float):
            f.write(f"{metric}: {val:.2f}\n")
        else:
            f.write(f"{metric}: {val}\n")
    f.write("="*40 + "\n")

/Users/ayraj/Desktop/video_captioning/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading ORIGINAL Pre-trained Model (Salesforce/blip-image-captioning-base) for Baseline Evaluation...


The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 473/473 [00:00<00:00, 51395.19it/s]
The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warnin

Found 2000 unique videos to evaluate.
Generating original AI captions for scoring...


Evaluating Videos:  14%|█▍        | 282/2000 [01:33<09:29,  3.02it/s]


KeyboardInterrupt: 